# Algothon 2026 — Strategy Research Notebook
> **Man Group × Imperial College Hackathon 2026**

This notebook provides an end-to-end research walkthrough of the portfolio strategy:

1. **Data Exploration** — prices, signals, volumes, yield curve
2. **Signal Construction** — TSMOM, XSMOM, multi-scale IVW combination
3. **Portfolio Optimisation** — Markowitz MVO with OAS shrinkage
4. **Backtest Analysis** — performance metrics, drawdowns, attribution
5. **Submission Generation** — compliant CSV for hackathon submission

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT / 'src' / 'python'))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from data.loader import DataLoader
from signals.momentum import MomentumEngine
from portfolio.optimizer import MVOOptimizer
from portfolio.risk import PerformanceAnalytics
from execution.engine import AlgothonEngine

# Plotly theme
import plotly.io as pio
pio.templates.default = 'plotly_dark'

print('All imports OK')

## 1. Data Exploration

In [ ]:
loader = DataLoader('data/sample')
ds = loader.load_all()
print(f'Prices   : {ds.prices.shape}')
print(f'Returns  : {ds.returns.shape}')
print(f'Signals  : {ds.signals.shape}')
print(f'Volumes  : {ds.volumes.shape}')
print(f'Date range: {ds.prices["date"].min()} → {ds.prices["date"].max()}')

In [ ]:
# Plot 1: Normalised price paths
instruments = [f'INSTRUMENT_{i}' for i in range(1, 11)]
prices_np = ds.prices.select(instruments).to_numpy().astype(float)
prices_norm = prices_np / prices_np[0]  # normalise to 1
dates = ds.prices['date'].to_list()

fig = go.Figure()
for i, inst in enumerate(instruments):
    fig.add_trace(go.Scatter(
        x=dates, y=prices_norm[:, i],
        mode='lines', name=inst, line=dict(width=1.5)
    ))
fig.update_layout(
    title='<b>Normalised Price Paths (2017–2026)</b>',
    xaxis_title='Date', yaxis_title='Cumulative Return (base=1)',
    height=500, hovermode='x unified'
)
fig.show()

In [ ]:
# Plot 2: Return distribution per instrument
ret_np = ds.returns.select(instruments).to_numpy().astype(float)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, inst in enumerate(instruments):
    axes[i].hist(ret_np[:, i], bins=80, density=True,
                 color='steelblue', alpha=0.8, edgecolor='none')
    axes[i].set_title(inst, fontsize=10)
    axes[i].set_xlabel('Log-return')
    vol = np.nanstd(ret_np[:, i]) * np.sqrt(252)
    axes[i].set_title(f'{inst}\nAnn.Vol={vol*100:.1f}%', fontsize=9)
plt.suptitle('Return Distributions (2017-2026)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Correlation heatmap
corr = np.corrcoef(ret_np.T)
fig = px.imshow(
    corr,
    labels=dict(color='Correlation'),
    x=instruments, y=instruments,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f',
    title='<b>Return Correlation Matrix</b>'
)
fig.update_layout(height=550)
fig.show()

## 2. Momentum Signal Construction

We compute multi-scale TSMOM and XSMOM signals then combine via IVW.

In [ ]:
engine = MomentumEngine(lookbacks=[4, 8, 16, 32], blend_alpha=0.7)
sigs = engine.compute(ds.returns, ds.signals)
print(f'TSMOM shape : {sigs.tsmom.shape}')
print(f'XSMOM shape : {sigs.xsmom.shape}')
print(f'Combined    : {sigs.combined.shape}')
print(f'EWM Vols    : {sigs.ewm_vols.shape}')

In [ ]:
# Plot 4: Signal heatmap (last 252 days)
recent = sigs.combined[-252:]
recent_dates = [str(d) for d in sigs.dates[-252:]]

fig = px.imshow(
    recent.T,
    x=recent_dates[::5], y=instruments,
    color_continuous_scale='RdYlGn',
    aspect='auto',
    title='<b>Combined Momentum Signal (Last 252 Days)</b>',
    labels={'color': 'Signal Strength'}
)
fig.update_xaxes(tickangle=45)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Plot 5: Signal timeseries for top 3 instruments
top3 = [0, 6, 8]  # INSTRUMENT_1, 7, 9
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=[instruments[i] for i in top3])
for row, i in enumerate(top3, 1):
    fig.add_trace(
        go.Scatter(x=list(sigs.dates), y=sigs.tsmom[:, i].tolist(),
                   name=f'{instruments[i]} TSMOM',
                   line=dict(color='royalblue', width=1)),
        row=row, col=1
    )
    fig.add_trace(
        go.Scatter(x=list(sigs.dates), y=sigs.combined[:, i].tolist(),
                   name=f'{instruments[i]} Combined',
                   line=dict(color='orange', width=1.5)),
        row=row, col=1
    )
fig.update_layout(height=700, title='<b>Momentum Signals: TSMOM vs Combined</b>')
fig.show()

## 3. Portfolio Optimisation

In [ ]:
opt = MVOOptimizer(risk_aversion=2.0, lw_shrinkage=True, max_weight=0.40)
ret_mat = ds.returns.select(instruments).to_numpy().astype(float)

# Raw signal weights
raw_w = engine.signal_to_weights(sigs, method='vol_parity', long_only=True)

# Rolling MVO optimisation
print('Running rolling MVO...')
opt_weights = opt.rolling_optimize(raw_w, ret_mat,
                                    rebalance_freq=5, warmup=63)
print(f'Optimal weights shape: {opt_weights.shape}')
print(f'Latest weights:\n{dict(zip(instruments, opt_weights[-1].round(4)))}')

In [ ]:
# Plot 6: Weight evolution over time
ret_dates = ds.returns['date'].to_list()

fig = go.Figure()
for i, inst in enumerate(instruments):
    fig.add_trace(go.Scatter(
        x=ret_dates, y=opt_weights[:, i].tolist(),
        stackgroup='one', name=inst,
        mode='lines', line=dict(width=0.5)
    ))
fig.update_layout(
    title='<b>Portfolio Weight Allocation Over Time (Stacked)</b>',
    xaxis_title='Date', yaxis_title='Weight',
    yaxis=dict(range=[0, 1], tickformat='.0%'),
    height=500, hovermode='x unified'
)
fig.show()

In [ ]:
# Plot 7: Covariance matrix (latest 252-day window)
cov = opt.estimate_covariance(ret_mat[-252:])
# Convert to correlation
std_diag = np.sqrt(np.diag(cov))
corr_mvo = cov / np.outer(std_diag, std_diag)

fig = px.imshow(
    corr_mvo,
    x=instruments, y=instruments,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1, text_auto='.2f',
    title='<b>LW-Shrunk Correlation (Latest 252-Day Window)</b>'
)
fig.update_layout(height=500)
fig.show()

## 4. Backtest Analysis

Compute comprehensive performance metrics for the full backtest period.

In [ ]:
pa = PerformanceAnalytics(risk_free_rate=0.05)
metrics = pa.compute(opt_weights, ret_mat)

print('=' * 50)
print('FULL BACKTEST PERFORMANCE METRICS')
print('=' * 50)
print(f'  Sharpe Ratio      : {metrics.sharpe:.4f}')
print(f'  Sortino Ratio     : {metrics.sortino:.4f}')
print(f'  Calmar Ratio      : {metrics.calmar:.4f}')
print(f'  Ann. Return       : {metrics.annualised_return*100:.2f}%')
print(f'  Ann. Volatility   : {metrics.annualised_vol*100:.2f}%')
print(f'  Max Drawdown      : {metrics.max_drawdown*100:.2f}%')
print(f'  CVaR (95%)        : {metrics.cvar_95*100:.2f}%')
print(f'  Hit Rate          : {metrics.hit_rate*100:.1f}%')
print(f'  Avg Daily Turnover: {metrics.avg_turnover*100:.2f}%')
print('=' * 50)

In [ ]:
# Plot 8: Cumulative returns with drawdown shading
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.7, 0.3],
                    subplot_titles=['Cumulative Wealth Index', 'Drawdown (%)'])

fig.add_trace(
    go.Scatter(x=ret_dates, y=metrics.cumulative_returns.tolist(),
               name='Portfolio', line=dict(color='#00D4AA', width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=ret_dates, y=(metrics.drawdowns * 100).tolist(),
               fill='tozeroy', name='Drawdown',
               line=dict(color='#FF4444', width=1),
               fillcolor='rgba(255,68,68,0.3)'),
    row=2, col=1
)

fig.update_yaxes(title_text='Wealth', row=1)
fig.update_yaxes(title_text='DD %', row=2)
fig.update_layout(height=600,
                  title='<b>Portfolio Cumulative Return and Drawdown</b>',
                  hovermode='x unified')
fig.show()

In [ ]:
# Plot 9: Rolling 252-day Sharpe ratio
roll_sharpe = pa.rolling_sharpe(opt_weights, ret_mat, window=252)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ret_dates, y=roll_sharpe.tolist(),
    mode='lines', name='Rolling Sharpe (252d)',
    line=dict(color='royalblue', width=2)
))
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.add_hline(y=1, line_dash='dot', line_color='green',
              annotation_text='Sharpe=1')
fig.update_layout(
    title='<b>Rolling 252-Day Sharpe Ratio</b>',
    xaxis_title='Date', yaxis_title='Sharpe',
    height=400
)
fig.show()

In [ ]:
# Plot 10: Monthly returns heatmap
port_ret_series = metrics.portfolio_returns
dates_np = np.array(ret_dates)

# Build monthly returns
import datetime
monthly = {}
for d, r in zip(dates_np, port_ret_series):
    key = (d.year, d.month)
    monthly[key] = monthly.get(key, 0.0) + r

years = sorted(set(k[0] for k in monthly))
months = list(range(1, 13))
mat = [[monthly.get((y, m), float('nan')) * 100 for m in months] for y in years]

fig = px.imshow(
    mat, x=['Jan','Feb','Mar','Apr','May','Jun',
            'Jul','Aug','Sep','Oct','Nov','Dec'],
    y=[str(y) for y in years],
    color_continuous_scale='RdYlGn',
    text_auto='.1f',
    title='<b>Monthly Returns (%)</b>',
    labels={'color': 'Return (%)'}
)
fig.update_layout(height=400)
fig.show()

## 5. Submission Generation

In [ ]:
eng = AlgothonEngine(
    data_dir='data/sample',
    team_name='algothon_team',
    risk_aversion=2.0,
    blend_alpha=0.7
)
metrics_run = eng.run_backtest(test_start='2025-01-01')
path = eng.generate_submission(round_number=1, output_dir='submissions')
print(f'\nSubmission saved: {path}')

# Display submission
import polars as pl
sub = pl.read_csv(path)
print(sub)

In [ ]:
# Plot 11: Submission weights pie chart
sub_df = pl.read_csv(path)
fig = px.pie(
    sub_df.to_pandas(),
    names='asset', values='weight',
    title='<b>Submission Portfolio Weights</b>',
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=500)
fig.show()